# 02 – Data preparation

Builds the analysis dataset (one row per PR) from the GitHub data collected by `src/github_collector.py`.

* **Size:** changed lines (additions + deletions) and changed files, for the whole PR.
* **Review activity:** reviews, inline review comments, distinct reviewers and discussion comments, each split into human (`User` account) and bot (`Bot` account).
* **Not counted as review:** activity by the PR author and by the agent's own working account on its PR (`copilot-swe-agent`, `devin-ai-integration`, `cursor`, `claude`, `chatgpt-codex-connector`).
* **Bots:** GitHub account type `Bot` or a bot-like login (`openshift-ci-robot`, `pytorchmergebot`, …).
* **Steering:** human discussion comments that address an agent (`@copilot …`) are counted separately. Formal reviews addressed to an agent stay reviews (they often carry code feedback) and are tracked for a robustness check.
* **Excluded PRs:** Google Jules PRs (too few repositories), human PRs that carry a trace of an agent writing code (e.g. Claude Code commit trailers), human PRs whose author has a bot-like login, and PRs with more than 100 reviews or comments (truncated by the API).

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
DATA = Path(os.environ.get("ESS_DATA", ROOT / "data"))        # override only for testing
RESULTS = Path(os.environ.get("ESS_RESULTS", ROOT / "results"))
import pandas as pd
import preprocessing

In [2]:
df, exclusions = preprocessing.build(DATA / "interim/sample.csv", DATA / "raw/details")
(DATA / "processed").mkdir(parents=True, exist_ok=True)
df.to_parquet(DATA / "processed/prs.parquet", index=False)
(RESULTS / "tables").mkdir(parents=True, exist_ok=True)
exclusions.to_csv(RESULTS / "tables/exclusions.csv", index=False)
exclusions

,step,remaining_prs
0,sampled PRs,335297
1,details fetched,335297
2,not Google Jules,333676
3,human PR without agent signature,309191
4,human PR authored by a User account without a ...,306692
5,reviews/comments not truncated (<=100),306512


## Composition of the dataset

In [3]:
print(f"{len(df):,} PRs in {df.repo.nunique()} repositories")
comp = df.groupby(["author_kind", df.agent.fillna("–")]).agg(PRs=("number", "size"), repositories=("repo", "nunique"))
comp.to_csv(RESULTS / "tables/dataset_composition.csv")
comp

306,512 PRs in 1248 repositories


PRs  repositories
author_kind agent                             
AI agent    Claude_Code    81259           975
            Copilot        34907           664
            Cursor          7513           253
            Devin           4818            92
            OpenAI_Codex   21482           635
Human       –             156533          1232

Repositories that contain both AI-agent and human PRs (only these inform the within-repository comparison):

In [4]:
both = df.groupby("repo").group.nunique().eq(2)
print(f"{both.sum()} of {len(both)} repositories have both groups; they hold {df.repo.isin(both[both].index).mean():.1%} of the PRs")

1226 of 1248 repositories have both groups; they hold 99.3% of the PRs


## Steering of agents

Share of PRs with at least one human message addressing an agent, and share of human reviews addressed to an agent (see Duma et al. 2026; manual check in `results/qa/steering_manual_check.csv`).

In [5]:
g = df.groupby(["author_kind", df.agent.fillna("–")])
pd.DataFrame({"PRs_with_steering_comment": g.steering_comments_human.apply(lambda s: (s > 0).mean()),
              "human_reviews_addressed_to_agent": g.steering_reviews_human.sum() / g.reviews_human.sum()}).round(3)

PRs_with_steering_comment  \
author_kind agent                                     
AI agent    Claude_Code                       0.007   
            Copilot                           0.286   
            Cursor                            0.007   
            Devin                             0.010   
            OpenAI_Codex                      0.004   
Human       –                                 0.007   

                          human_reviews_addressed_to_agent  
author_kind agent                                           
AI agent    Claude_Code                              0.000  
            Copilot                                  0.056  
            Cursor                                   0.000  
            Devin                                    0.000  
            OpenAI_Codex                             0.000  
Human       –                                        0.000